In [1]:
#may need to install these packages first

#!uv pip install hdbscan
#!uv pip install geopandas

## Settings

In [2]:
import os
import glob
import plotly.express as px
import pandas as pd
import numpy as np
import polars as pl
import hdbscan

import warnings
from birds.source_data.nabbp import LookupTables #, DataTables
#from birds.source_data.iucn import IUCNDataTables
from birds.settings import load_settings, get_relative_path
from birds.transform_nabbp_to_band_agg import get_nabbp_data_files,load_nabbp_data, write_parquet_file

from sklearn.utils import resample

import geopandas as gpd
import plotly.graph_objects as go

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import silhouette_score
import altair as alt
from hdbscan.validity import validity_index

Source NABBP data path: data/source_data/NABBP-2025/grpdata
Destination path for agg parquet files: data/augmented_data/band_agg


In [3]:
warnings.filterwarnings('ignore', message='.*force_all_finite.*')

In [4]:
# load index with redlist

settings = load_settings()
lookup = LookupTables()

In [5]:
nabbp_data_path = get_relative_path(settings.nabbp_data_path)
augmented_data_path =get_relative_path(settings.augmented_data_base_path)
dest_data_path = get_relative_path(settings.augmented_atrisk_events)
model_path = get_relative_path(settings.model_path)

print(dest_data_path)
print(nabbp_data_path)
print(augmented_data_path)
print(model_path)

data/augmented_data/at_risk_events
data/source_data/NABBP-2025/grpdata
data/augmented_data
data/models


## Load Data

In [6]:
def load_atrisk_data_files(nfiles):
    data_dir = get_relative_path(get_relative_path(settings.augmented_atrisk_events))
    print(data_dir)

    all_files = sorted(glob.glob(os.path.join(data_dir, "atrisk_*.parquet")))

    if nfiles > 0:
        all_files = all_files[-nfiles:]
    
    print(f"Loading {len(all_files)} files from {data_dir}")
    df = pl.concat([pl.read_parquet(f) for f in all_files])
    return df


riskdf = load_atrisk_data_files(nfiles=58)


data/augmented_data/at_risk_events
Loading 57 files from data/augmented_data/at_risk_events


In [7]:
riskdf.shape

(5102864, 6)

In [8]:

def filter_contiguous_us(df):

    df = df.to_pandas()
    mask = (
        (df['lat_dd'] >= 25.0) &   # Just above Florida Keys
        (df['lat_dd'] <= 49.0) &   # Just below Canadian border
        (df['lon_dd'] >= -124.5) & # Pacific Northwest coast
        (df['lon_dd'] <= -67.0)    # Maine coast
    )
    
    return df[mask]


riskdf = filter_contiguous_us(riskdf)

In [9]:
riskdf.shape

(4010056, 6)

## Prepare Data for Clustering

A choice was made to segment clustering by decade in order to compare clusters over time and At-Risk vs Not-At-Risk.  

In [10]:
def prep_for_clustering(df: pd.DataFrame) -> (pd.DataFrame, pd.DataFrame):
    
    # add event_decade based on event_year
    #df = df.with_columns((pl.col('event_year')//10*10).alias('event_decade'))
    df['event_decade'] = (df['event_year']//10*10)
    
    # split to at-risk and not-at-risk
    # convert below to pandas for resampling
    atriskdf = df[df['atRisk'] == True]
    notatriskdf = df[df['atRisk'] == False]

    # downsample not-at-risk to match at-risk
    if len(atriskdf) < len(notatriskdf):
        notatriskdf = resample(notatriskdf, replace=False, n_samples=len(atriskdf), random_state=42)
    

    return atriskdf, notatriskdf

atriskdf, notatriskdf = prep_for_clustering(riskdf)

In [11]:
# sanity check data, should be similarly sized and have no null or 0 lat/lon
print(f"At-risk rows: {len(atriskdf)}, Not-at-risk rows: {len(notatriskdf)}")
print(f"At-risk rows with null lat/lon: {atriskdf['lat_dd'].isnull().sum() + atriskdf['lon_dd'].isnull().sum()}, Not-at-risk rows with null lat/lon: {notatriskdf['lat_dd'].isnull().sum() + notatriskdf['lon_dd'].isnull().sum()}")
print(f"At-risk rows with 0 lat/lon: {((atriskdf['lat_dd'] == 0).sum() + (atriskdf['lon_dd'] == 0).sum())}, Not-at-risk rows with 0 lat/lon: {((notatriskdf['lat_dd'] == 0).sum() + (notatriskdf['lon_dd'] == 0).sum())}")  


At-risk rows: 50368, Not-at-risk rows: 50368
At-risk rows with null lat/lon: 0, Not-at-risk rows with null lat/lon: 0
At-risk rows with 0 lat/lon: 0, Not-at-risk rows with 0 lat/lon: 0


## Find Optimal Hyper parameters

Average DBCV across decades and Atrisk status to find best average hyperparameters

In [12]:
def evaluate_hdbscan(df, min_cluster_size, min_samples):
    """Simple evaluation function for HDBSCAN parameters"""
    coords = np.radians(df[['lat_dd', 'lon_dd']].values)
    
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric='haversine'
    )
    labels = clusterer.fit_predict(coords)
    


    # metrics
    mask = labels != -1
    silhouette = None
    dbcv = None

    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    pct_clustered = ((labels != -1).sum() / len(labels)) * 100
    
    

    if mask.sum() > 1 and n_clusters > 1:
        silhouette = silhouette_score(coords[mask], labels[mask],  metric='haversine')
        
        
    # DBCV requires at least 2 clusters (can include noise as a "cluster")
    # https://towardsdatascience.com/tuning-with-hdbscan-149865ac2970/
    
    if n_clusters >= 1:
        dbcv = validity_index(coords, labels)
    
    return {
        'min_cluster_size': min_cluster_size,
        'min_samples': min_samples,
        'n_clusters': n_clusters,
        'n_noise': n_noise,
        'pct_clustered': pct_clustered,
        'silhouette': silhouette,
        'dbcv': dbcv
    }

In [13]:
# run grid search to find optimal parms if cannot load previous results

results_file = os.path.join(model_path, 'hdbscan_grid_search_results.pkl')
print (f"Checking for previous results at {results_file}...")
if os.path.exists(results_file):
    print(f"Loading previous results from {results_file}")
    results_df = pd.read_pickle(results_file)
else:
    print("No previous results found, running grid search...")
    results_df = None

    param_grid = {
        'min_cluster_size': [5, 10, 15, 20, 30],
        'min_samples': [5, 10, 15, 20]
    }

    decades = [1960, 1970, 1980, 1990, 2000, 2010, 2020]
    risk_types = ['at-risk', 'not-at-risk']

    all_results = []

    print("Running grid search across all decades and risk types...")
    for decade in decades:
        for risk_type in risk_types:
            if risk_type == 'at-risk':
                df = atriskdf[atriskdf['event_decade'] == decade].copy()
            else:
                df = notatriskdf[notatriskdf['event_decade'] == decade].copy()
            
            if len(df) == 0:
                continue
            
            print(f"\n{risk_type} {decade}s ({len(df)} points)")
            
            for params in ParameterGrid(param_grid):
                result = evaluate_hdbscan(df, **params)
                result['decade'] = decade
                result['risk_type'] = risk_type
                all_results.append(result)

    results_df = pd.DataFrame(all_results)

    print("Saving grid search results...")
    results_df.to_pickle(results_file)


Checking for previous results at data/models/hdbscan_grid_search_results.pkl...
Loading previous results from data/models/hdbscan_grid_search_results.pkl


In [14]:
# average dbcv results across min_cluster_size and min_samples
average_dbcv = results_df.groupby(['min_cluster_size', 'min_samples']).agg({'dbcv': 'mean'}).reset_index()
average_dbcv.head()

,min_cluster_size,min_samples,dbcv
0,5,5,0.331196
1,5,10,0.290820
2,5,15,0.276601
3,5,20,0.251846
4,10,5,0.264307


In [15]:
# heatmap plot of dbcv by min_cluster_size and min_samples
chart = alt.Chart(average_dbcv).mark_rect().encode(
    x=alt.X('min_cluster_size:O', title='Min Cluster Size'),
    y=alt.Y('min_samples:O', title='Min Samples'),
    color=alt.Color('dbcv:Q', title='Average DBCV', scale=alt.Scale(scheme='blues'))
).properties(
    title='Average DBCV by HDBSCAN Parameters',
    width=400,
    height=300
)
chart.show()


alt.Chart(...)

In [16]:
average_dbcv

,min_cluster_size,min_samples,dbcv
0,5,5,0.331196
1,5,10,0.290820
2,5,15,0.276601
3,5,20,0.251846
4,10,5,0.264307
5,10,10,0.270451
6,10,15,0.260888
7,10,20,0.262262
8,15,5,0.223676
9,15,10,0.252408


## Cluster all decades and atrisk status variations with optimal Parameters

Best parameters based on Grid Search the optimal parameers are min_cluster_size = 5 and min_samples = 5

In [17]:
def cluster_data(df: pd.DataFrame, min_cluster_size=5, min_samples=5) -> pd.DataFrame:

    # convert to radians for haversine distance clustering
    df['lat_dd_radians'] = np.radians(df['lat_dd'])
    df['lon_dd_radians'] = np.radians(df['lon_dd'])

    # cluster on lat_dd and lon_dd
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, metric='haversine')
    df['cluster'] = clusterer.fit_predict(df[['lat_dd_radians', 'lon_dd_radians']])

    # drop rows with cluster = -1 (noise)
    df = df[df['cluster'] != -1]
    return df



In [18]:
# iterate through decades and cluster each decade separately
# separate by decade in order to compare changes over time
decades = sorted(atriskdf['event_decade'].unique())

clustered_atrisk_decade_dfs = []
clustered_notatrisk_decade_dfs = []

for decade in decades:
    # atrisk data
    print(f'clustering atrisk decade: {decade}s')
    decade_df = atriskdf[atriskdf['event_decade'] == decade].copy()
    decade_clustered = cluster_data(decade_df)
    clustered_atrisk_decade_dfs.append(decade_clustered)
    # notatrisk data
    print(f'clustering notatrisk decade: {decade}s')
    not_decade_df = notatriskdf[notatriskdf['event_decade'] == decade].copy()
    not_decade_clustered = cluster_data(not_decade_df)
    clustered_notatrisk_decade_dfs.append(not_decade_clustered)

atrisk_clustered = pd.concat(clustered_atrisk_decade_dfs)
notatrisk_clustered = pd.concat(clustered_notatrisk_decade_dfs)


clustering atrisk decade: 1960s
clustering notatrisk decade: 1960s
clustering atrisk decade: 1970s
clustering notatrisk decade: 1970s
clustering atrisk decade: 1980s
clustering notatrisk decade: 1980s
clustering atrisk decade: 1990s
clustering notatrisk decade: 1990s
clustering atrisk decade: 2000s
clustering notatrisk decade: 2000s
clustering atrisk decade: 2010s
clustering notatrisk decade: 2010s
clustering atrisk decade: 2020s
clustering notatrisk decade: 2020s


In [19]:

def plot_plotly_heatmap(df: pd.DataFrame, title: str, height=600, width=600):
    fig = px.density_map(df, lat='lat_dd', lon='lon_dd', 
                        radius=5,
                        #center=dict(lat=45, lon=-100),
                        center=dict(lat=40, lon=-96),
                        zoom=2.5)

    fig.update_layout(title=title, 
                      height=height, 
                      width=width,
                      title_font_size=24,
                      margin={"r":0,"t":40,"l":0,"b":0}
                      )
    #fig.show()

    return fig

In [20]:
def plot_clustered_heatmap(df: pd.DataFrame, title: str, height=600, width=600):

    # normalize to match heatmap scale
    df_plot = df.copy()
    min_count = df['count'].min()
    max_count = df['count'].max()
    df_plot['normalized_count'] = (df['count'] - min_count) / (max_count - min_count)
    

    fig = px.density_map(df_plot, lat='centroid_lat', lon='centroid_lon', 
                        radius=5,
                        center=dict(lat=40, lon=-96),
                        zoom=2.5,
                        z='normalized_count',
                        range_color=[0, df_plot['normalized_count'].quantile(0.75)]  #better align color scale with heatmap
                        )

    fig.update_layout(title=title, 
                      height=height, 
                      width=width,
                      title_font_size=24,
                      margin={"r":0,"t":40,"l":0,"b":0}
                      )
    #fig.show()
    fig.update_coloraxes(colorbar_title_text='Count')
    return fig

In [21]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate distance between two points in kilometers (roughly based on https://community.esri.com/t5/coordinate-reference-systems-blog/distance-on-a-sphere-the-haversine-formula/ba-p/902128)
    """
    R = 6371  # Earth's radius in km
    
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c



def calculate_cluster_metrics(df, cluster_col, lat_col='lat_dd', lon_col='lon_dd', percentile=95):
    """
    Take a dataframe including geographic clusters,
    calculate the centroid of each cluster (mean lat/lon),
    then calulate the specified percentile distance from centroid to all points in the cluster.
    Return a dataframe with cluster_id, centroid lat/lon, and radius at the specified percentile
    also return count of points in each cluster.
    """
    df= df.copy()
    results = []
    
    for cluster_id in df[cluster_col].unique():
        cluster_data = df[df[cluster_col] == cluster_id]
        
        centroid_lat = cluster_data[lat_col].mean()
        centroid_lon = cluster_data[lon_col].mean()
        count = len(cluster_data)
        
        distances = [
            haversine_distance(centroid_lat, centroid_lon, row[lat_col], row[lon_col])
            for _, row in cluster_data.iterrows()
        ]
        
        radius = np.percentile(distances, percentile)
        
        results.append({
            'cluster_id': cluster_id,
            'centroid_lat': centroid_lat,
            'centroid_lon': centroid_lon,
            f'p{percentile}_radius_km': radius,
            'count': count
        })
    
    return pd.DataFrame(results)


#atrisk2020s_w_radius =  calculate_percentile_radius(atrisk2020s, 'cluster', percentile=95)

In [22]:
#atrisk2020s_w_radius.columns

In [23]:

def plot_uncertainty_map(df, decade, risktype):
    """Plot cluster centroids with uncertainty circles using Plotly."""
    
    # create geopandas points from centroids and buffer for uncertainty circles
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.centroid_lon, df.centroid_lat),
        crs='EPSG:4326'
    )
    gdf['circle'] = gdf.geometry.buffer(df['p95_radius_km'] / 111) ## rougly convert km to degrees
    
    fig = go.Figure()
    
    # circles
    for idx, row in gdf.iterrows():
        x, y = row['circle'].exterior.coords.xy
        fig.add_trace(go.Scattergeo(
            lon=list(x),
            lat=list(y),
            mode='lines',
            line=dict(width=1, color='darkblue'), 
            fill='toself',
            fillcolor='lightblue',
            name=f'Cluster {row["cluster_id"]}',
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # centroids
    fig.add_trace(go.Scattergeo(
        lon=gdf.geometry.x,
        lat=gdf.geometry.y,
        mode='markers',
        marker=dict(size=5, color='blue', line=dict(width=1, color='white')),
        text=gdf['cluster_id'],
        showlegend=False
    ))
    
    fig.update_layout(
        title=f'{risktype} {decade}s Centroids 95 %ile Radius',
        title_font_size=24,
        geo=dict(
            projection_type='mercator',  
            showland=True,
            showcountries=True,
            center=dict(lat=40, lon=-96),
            lataxis_range=[20, 55],  # Adjust latitude range to focus on US
            lonaxis_range=[-130, -60]  # Adjust longitude range to focus on US
        ),
        height=400,
        width=600,
        margin={"r":0,"t":40,"l":0,"b":0}
    )
    
    return fig

In [24]:
def plot_clusters_sizes(df,decade,risktype):
    """Plot clusters on a US map with circle size and color based on count."""
    

    if risktype == 'At-Risk':
        colortype = 'lightcoral'
    else:
        colortype = 'dodgerblue'

    fig = px.scatter_map(
    #fig = px.scatter_mapbox(
        df,
        lat='centroid_lat',
        lon='centroid_lon',
        size='count',
        color_discrete_sequence=[colortype],
        #hover_data=['cluster_id', 'count'],
        zoom=2.5,
        #center=dict(lat=39.8283, lon=-98.5795),
        center=dict(lat=40, lon=-96),
        #mapbox_style='open-street-map',
        map_style='open-street-map',
        height=600,
        width=600,
        #size_max=50, 
        title=f'{risktype} {decade}s Cluster Sizes'
    )

    fig.update_layout(
        title_x=0.5, 
        title_font_size=24,
        margin={"r":0,"t":40,"l":0,"b":0})
    
    return fig



In [25]:
# create separate dataframes for each decade from 1990 to 2020 and at-ris and not-at-risk
decades = [1960,1970,1980,1990, 2000, 2010, 2020]
risktypes = ['At-Risk', 'Not-At-Risk']
decade_risktype_dfs = {}
for decade in decades:
    for risktype in risktypes:
        if risktype == 'At-Risk':
            df = atrisk_clustered[atrisk_clustered['event_decade'] == decade]
        else:
            df = notatrisk_clustered[notatrisk_clustered['event_decade'] == decade]
        
        if not df.empty:
            #clustered_df = cluster_data(df)
            if not df.empty:
                centroids_df = calculate_cluster_metrics(df, 'cluster', percentile=95)
                decade_risktype_dfs[(decade, risktype)] = centroids_df
                print(f"Processed {risktype} data for {decade}s: {len(df)} points, {len(centroids_df)} clusters")
            else:
                print(f"No clusters found for {risktype} in {decade}s")
        else:
            print(f"No data for {risktype} in {decade}s")
            
        

Processed At-Risk data for 1960s: 8047 points, 381 clusters
Processed Not-At-Risk data for 1960s: 3640 points, 293 clusters
Processed At-Risk data for 1970s: 8001 points, 333 clusters
Processed Not-At-Risk data for 1970s: 4354 points, 341 clusters
Processed At-Risk data for 1980s: 3453 points, 174 clusters
Processed Not-At-Risk data for 1980s: 3874 points, 290 clusters
Processed At-Risk data for 1990s: 1955 points, 96 clusters
Processed Not-At-Risk data for 1990s: 4620 points, 378 clusters
Processed At-Risk data for 2000s: 3812 points, 160 clusters
Processed Not-At-Risk data for 2000s: 6570 points, 576 clusters
Processed At-Risk data for 2010s: 13645 points, 397 clusters
Processed Not-At-Risk data for 2010s: 7282 points, 518 clusters
Processed At-Risk data for 2020s: 5233 points, 224 clusters
Processed Not-At-Risk data for 2020s: 4795 points, 312 clusters


In [26]:
plot_plotly_heatmap(atrisk_clustered[atrisk_clustered['event_decade']==2020], 'At-Risk Species 2020s Cluster Centroids')

In [27]:
plot_clustered_heatmap(decade_risktype_dfs[(2020,'At-Risk')], 'At-Risk Species 2020s Plotly Heatmap')


In [28]:
plot_uncertainty_map(decade_risktype_dfs[(2020,'Not-At-Risk')],2020,'Not-At-Risk')

/tmp/ipykernel_56032/134574207.py:10: UserWarning:

Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.




In [29]:
plot_uncertainty_map(decade_risktype_dfs[(2020,'At-Risk')],2020,'At-Risk')


/tmp/ipykernel_56032/134574207.py:10: UserWarning:

Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.




In [30]:
for dfkey in decade_risktype_dfs.keys():
    decade, risktype = dfkey
    df = decade_risktype_dfs[dfkey]
    print(dfkey,len(df))
    fig = plot_clusters_sizes(decade_risktype_dfs[(decade,risktype)],decade,risktype)
    fig.show()

(1960, 'At-Risk') 381


(1960, 'Not-At-Risk') 293


(1970, 'At-Risk') 333


(1970, 'Not-At-Risk') 341


(1980, 'At-Risk') 174


...

(2000, 'Not-At-Risk') 576


(2010, 'At-Risk') 397


(2010, 'Not-At-Risk') 518


(2020, 'At-Risk') 224


(2020, 'Not-At-Risk') 312
